# 🤖 AI Agents Roadmap — Phase 1: Prompt Engineering Masterclass

> **Dành cho:** Python Developer có kinh nghiệm | **Thời gian:** 3–5 ngày thực hành
> 
> **Mục tiêu:** Nắm vững các kỹ thuật Prompt Engineering để xây dựng AI Agents production-ready

---

## 🗺️ Roadmap Tổng Quan

```
Phase 1 (Tuần 1–2) ← BẠN ĐANG Ở ĐÂY
├── LLM APIs (OpenAI · Anthropic · Gemini)
├── Prompt Engineering ← NỘI DUNG NOTEBOOK NÀY
│   ├── Zero/Few-shot · Chain-of-Thought · Role Prompting
│   └── ReAct · Self-Critique · Meta-Prompting
├── Tokens & Context
├── Embeddings
└── Structured Output

Phase 2 → Tool Use · ReAct · RAG · Vector DB
Phase 3 → LangChain · LangGraph · CrewAI · Production
Phase 4 → Multi-agent · Planning · Human-in-the-loop
Phase 5 → Safety · Fine-tuning · Frontier Research
```

## 📚 Nội Dung Notebook

| # | Chủ đề | Kỹ thuật |
|---|--------|----------|
| 1 | Setup & OpenAI API Basics | Authentication, Models, Parameters |
| 2 | Zero-shot / Few-shot Prompting | Examples, Dynamic Selection |
| 3 | Chain-of-Thought (CoT) | Zero-shot CoT, Few-shot CoT, Self-Consistency, ToT |
| 4 | Role & System Prompt Design | Anatomy, Persona Engineering, Multi-agent |
| 5 | Structured Output & JSON Mode | Schemas, Pydantic Validation |
| 6 | Prompt Templates & Chaining | Jinja2, Prompt Chains |
| 7 | ReAct Pattern | Reason → Act → Observe loop |
| 8 | Advanced Techniques | Meta-Prompting, Self-Critique, Reflection |
| 9 | Prompt Evaluation & A/B Testing | Metrics, Evaluator Framework |
| 10 | Production: Smart Prompt Manager | Versioning, Caching, Token Counting |

---
## 🔧 Phần 1: Setup & OpenAI API Basics

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install openai tiktoken jinja2 pydantic --quiet

In [ ]:
import os
import json
import time
import asyncio
import hashlib
import statistics
from typing import Optional, Callable
from dataclasses import dataclass, field

import tiktoken
from jinja2 import Template
from pydantic import BaseModel, Field
from openai import OpenAI, AsyncOpenAI

# ============================================================
# ⚙️ CẤU HÌNH: Nhập API Key của bạn vào đây
# ============================================================
# Cách 1 (khuyên dùng): Set environment variable
#   export OPENAI_API_KEY='sk-...'
# Cách 2: Nhập trực tiếp (chỉ dùng để học, KHÔNG commit lên git)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "sk-YOUR_KEY_HERE")

client = OpenAI(api_key=OPENAI_API_KEY)
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# Model mặc định — có thể đổi thành gpt-4o, gpt-4-turbo, gpt-3.5-turbo
DEFAULT_MODEL = "gpt-4o-mini"   # Rẻ nhất, phù hợp để học
SMART_MODEL   = "gpt-4o"        # Mạnh hơn cho các bài tập nâng cao

print("✅ OpenAI client đã khởi tạo!")
print(f"   Default model : {DEFAULT_MODEL}")
print(f"   Smart model   : {SMART_MODEL}")

In [ ]:
# ============================================================
# 🛠️ Helper function dùng xuyên suốt notebook
# ============================================================

def chat(
    user_message: str,
    system_message: str = "You are a helpful AI assistant.",
    model: str = DEFAULT_MODEL,
    temperature: float = 0.1,
    max_tokens: int = 1000,
    verbose: bool = True
) -> str:
    """
    Wrapper đơn giản cho OpenAI Chat API.
    Trả về text response và (tuỳ chọn) in thông tin usage.
    """
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_message}
        ]
    )
    result = response.choices[0].message.content
    if verbose:
        usage = response.usage
        print(f"📊 Tokens — Input: {usage.prompt_tokens} | Output: {usage.completion_tokens} | Total: {usage.total_tokens}")
        print("-" * 60)
        print(result)
    return result


def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """Đếm số token của một đoạn text."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))


print("✅ Helper functions đã sẵn sàng!")

# Test nhanh
test_text = "Hello, I am learning AI Agents!"
print(f"Token count của '{test_text}': {count_tokens(test_text)} tokens")

### 💡 LLM như một hàm toán học

```python
# Mental model quan trọng nhất:
output = f(system_prompt, conversation_history, user_input)

# Prompt Engineering = tối ưu hoá TẤT CẢ các tham số đầu vào
# để đạt được output chất lượng cao nhất
```

**Các tham số quan trọng của OpenAI API:**

| Tham số | Ý nghĩa | Giá trị khuyên dùng |
|---------|---------|--------------------|
| `temperature` | Độ ngẫu nhiên (0=deterministic, 2=rất random) | 0.0–0.2 cho production |
| `top_p` | Nucleus sampling (thay thế temperature) | 0.9 nếu dùng thay temperature |
| `max_tokens` | Giới hạn độ dài output | Đặt hợp lý, không để mặc định |
| `frequency_penalty` | Giảm lặp từ (0–2) | 0.3–0.5 cho text sáng tạo |
| `presence_penalty` | Khuyến khích chủ đề mới (0–2) | 0.1–0.3 |
| `seed` | Tái tạo output deterministic | Bất kỳ integer |
| `response_format` | Bắt buộc JSON output | `{"type": "json_object"}` |

In [ ]:
# 🧪 THỰC HÀNH: So sánh temperature
prompt = "Viết một câu mô tả về AI Agents."

print("=" * 60)
print("Temperature = 0.0 (deterministic):")
_ = chat(prompt, temperature=0.0)

print("\n" + "=" * 60)
print("Temperature = 1.0 (creative):")
_ = chat(prompt, temperature=1.0)

# 🎯 Bài tập: Chạy cell này nhiều lần và quan sát sự khác biệt
# Temperature 0.0 → Luôn ra kết quả giống nhau
# Temperature 1.0 → Mỗi lần ra kết quả khác nhau

---
## 🎯 Phần 2: Zero-shot, One-shot & Few-shot Prompting

In [ ]:
# ============================================================
# 2.1 ZERO-SHOT PROMPTING
# Không có ví dụ — model dựa hoàn toàn vào kiến thức pre-training
# ============================================================

zero_shot_prompt = """
Classify the sentiment of the following review:

Review: "Sản phẩm tệ, không đáng tiền, giao hàng chậm"
Sentiment (POSITIVE/NEGATIVE/NEUTRAL):
"""

print("🔍 ZERO-SHOT:")
_ = chat(zero_shot_prompt)

In [ ]:
# ============================================================
# 2.2 FEW-SHOT PROMPTING — Anatomy of a Good Few-shot Prompt
# Cung cấp ví dụ để dạy model format và pattern mong muốn
# ============================================================

system_prompt = "You are a sentiment analysis expert for an e-commerce platform."

few_shot_prompt = """
# TASK: Phân tích cảm xúc của review sản phẩm
# FORMAT: Label (POSITIVE/NEGATIVE/NEUTRAL) + Score (0.0–1.0) + Reason (1 câu)

---
# Example 1:
Review: "Giao hàng nhanh, đóng gói cẩn thận, sản phẩm đúng như mô tả"
Label: POSITIVE
Score: 0.95
Reason: Đề cập tích cực về 3 yếu tố quan trọng: tốc độ, bao bì, và chất lượng

# Example 2:
Review: "Chất lượng tạm được nhưng giá hơi cao so với thị trường"
Label: NEUTRAL
Score: 0.45
Reason: Có điểm tốt (chất lượng) nhưng cũng có điểm tiêu cực (giá cả)

# Example 3:
Review: "Hàng kém chất lượng, shop không hỗ trợ, rất thất vọng"
Label: NEGATIVE
Score: 0.05
Reason: Nhiều vấn đề nghiêm trọng: chất lượng kém và dịch vụ tệ

---
# Now analyze:
Review: "Thiết kế đẹp nhưng pin không đúng như quảng cáo, hơi thất vọng"
Label:
Score:
Reason:
"""

print("🔍 FEW-SHOT (3 examples):")
_ = chat(few_shot_prompt, system_message=system_prompt)

In [ ]:
# ============================================================
# 2.3 DYNAMIC FEW-SHOT SELECTION
# Kỹ thuật nâng cao: chọn examples phù hợp nhất với input hiện tại
# Trong production, dùng embedding similarity để chọn examples
# ============================================================

EXAMPLE_BANK = [
    {
        "review": "Pin trâu, camera đẹp, màn hình sắc nét, rất đáng mua",
        "category": "phone",
        "label": "POSITIVE", "score": 0.92
    },
    {
        "review": "Máy tính xách tay tản nhiệt kém, hay nóng khi dùng lâu",
        "category": "laptop",
        "label": "NEGATIVE", "score": 0.20
    },
    {
        "review": "Tai nghe âm thanh tốt, kết nối ổn định, pin dùng được khoảng 8 tiếng",
        "category": "earphone",
        "label": "POSITIVE", "score": 0.80
    },
    {
        "review": "Điện thoại oke nhưng màn hình hay lag khi chơi game nặng",
        "category": "phone",
        "label": "NEUTRAL", "score": 0.50
    },
]


def select_examples(query: str, bank: list, n: int = 2) -> list:
    """
    Simple keyword-based selection (production: dùng embedding cosine similarity).
    Chọn n examples liên quan nhất đến query.
    """
    # Trong thực tế: score = cosine_similarity(embed(query), embed(example['review']))
    # Đây là phiên bản simplified dùng keyword matching
    keywords = set(query.lower().split())
    scored = []
    for ex in bank:
        ex_words = set(ex["review"].lower().split())
        overlap = len(keywords & ex_words)
        scored.append((overlap, ex))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [ex for _, ex in scored[:n]]


def build_dynamic_few_shot_prompt(query: str, n_examples: int = 2) -> str:
    examples = select_examples(query, EXAMPLE_BANK, n_examples)
    prompt_parts = ["# Phân tích sentiment của review sản phẩm điện tử\n"]
    for i, ex in enumerate(examples, 1):
        prompt_parts.append(f"# Example {i}:")
        prompt_parts.append(f"Review: \"{ex['review']}\"")
        prompt_parts.append(f"Label: {ex['label']}")
        prompt_parts.append(f"Score: {ex['score']}\n")
    prompt_parts.append("# Now analyze:")
    prompt_parts.append(f"Review: \"{query}\"")
    prompt_parts.append("Label:\nScore:")
    return "\n".join(prompt_parts)


# Test dynamic selection
test_review = "Điện thoại đẹp nhưng pin yếu, chỉ dùng được nửa ngày"
dynamic_prompt = build_dynamic_few_shot_prompt(test_review)

print("📝 Dynamic Few-shot Prompt được tạo:")
print(dynamic_prompt)
print("\n" + "=" * 60)
print("🤖 Kết quả:")
_ = chat(dynamic_prompt)

### ✅ Best Practices cho Few-shot Prompting

| Nguyên tắc | Chi tiết |
|-----------|----------|
| **Quality > Quantity** | 3–5 examples xuất sắc > 10 examples trung bình |
| **Đa dạng examples** | Cover các edge cases và variations quan trọng |
| **Nhất quán format** | Tất cả examples phải có cùng cấu trúc |
| **Recency bias** | Example cuối có ảnh hưởng lớn nhất |
| **Task trước, examples sau** | Mô tả task trước khi đưa examples |
| **Dynamic selection** | Chọn examples phù hợp nhất với input (↑15–20% accuracy) |

---
## 🧠 Phần 3: Chain-of-Thought (CoT) Prompting

In [ ]:
# ============================================================
# 3.1 ZERO-SHOT CoT — "Let's think step by step"
# Wei et al., 2022: Tăng accuracy từ ~18% lên ~57% trên arithmetic tasks
# ============================================================

# Bài toán tính chi phí token — rất thực tế trong AI Agent development!
problem = """
Một AI Agent cần xử lý 1,000 requests mỗi ngày.
Mỗi request tiêu tốn trung bình 800 tokens (input + output).
GPT-4o-mini có giá: $0.00015/1K input tokens, $0.0006/1K output tokens.
Giả sử tỷ lệ input:output = 70:30.
Tổng chi phí mỗi tháng (30 ngày) là bao nhiêu USD?
"""

# Không dùng CoT
print("❌ WITHOUT CoT:")
_ = chat(problem)

print("\n" + "=" * 60)
print("✅ WITH Zero-shot CoT:")
_ = chat(problem + "\n\nHãy suy nghĩ từng bước trước khi đưa ra câu trả lời.")

In [ ]:
# ============================================================
# 3.2 FEW-SHOT CoT — Dạy reasoning pattern qua examples
# ============================================================

cot_few_shot = """
Giải quyết các bài toán về AI Agent performance:

---
Q: Nếu agent gọi 3 tools song song, mỗi tool mất 2 giây,
   nhưng 1 tool timeout sau 5 giây, tổng thời gian là bao nhiêu?
A: Hãy suy nghĩ từng bước:
   1. 3 tools chạy song song → thời gian = max(thời gian từng tool)
   2. Tool 1: 2s, Tool 2: 2s, Tool 3: timeout = 5s
   3. max(2, 2, 5) = 5 seconds
   4. Thêm retry logic overhead: +2s
   → Kết quả: ~7s với retry, hoặc 5s nếu fail-fast

---
Q: Một agent có context window 128K tokens. Mỗi turn conversation
   trung bình 500 tokens. Agent cần lưu system prompt 2000 tokens
   và tool definitions 1000 tokens. Tối đa bao nhiêu turns?
A: Hãy suy nghĩ từng bước:
   1. Context window: 128,000 tokens
   2. Fixed overhead: system_prompt(2000) + tools(1000) = 3,000 tokens
   3. Còn lại cho conversation: 128,000 - 3,000 = 125,000 tokens
   4. Mỗi turn: 500 tokens
   5. Tối đa turns: 125,000 / 500 = 250 turns
   → Kết quả: 250 turns (thực tế nên giới hạn ~200 để có buffer)

---
Q: Agent dùng RAG pipeline: embedding 1000 docs, mỗi doc 512 tokens.
   Embedding model ada-002 giá $0.0001/1K tokens.
   Chi phí index toàn bộ corpus là bao nhiêu?
A: Hãy suy nghĩ từng bước:
"""

print("🔍 FEW-SHOT CoT:")
_ = chat(cot_few_shot, temperature=0.1)

In [ ]:
# ============================================================
# 3.3 SELF-CONSISTENCY — Majority voting để tăng accuracy
# Chạy cùng prompt nhiều lần với temperature > 0, lấy đáp án phổ biến nhất
# ============================================================

from collections import Counter


def self_consistency(
    prompt: str,
    n_samples: int = 5,
    temperature: float = 0.7,
    extract_final_answer: Callable = None
) -> dict:
    """
    Chạy prompt n lần, trả về majority answer.
    Phù hợp cho bài toán có đáp án đúng/sai rõ ràng.
    Chi phí: n × đơn giá — dùng khi cần accuracy cao.
    """
    answers = []
    full_responses = []

    print(f"🔄 Chạy {n_samples} samples với temperature={temperature}...")

    for i in range(n_samples):
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=temperature,
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        full_text = response.choices[0].message.content
        full_responses.append(full_text)

        # Extract final answer (dòng cuối cùng thường là answer)
        if extract_final_answer:
            answer = extract_final_answer(full_text)
        else:
            answer = full_text.strip().split("\n")[-1].strip()

        answers.append(answer)
        print(f"  Sample {i+1}: {answer[:80]}..." if len(answer) > 80 else f"  Sample {i+1}: {answer}")

    # Majority vote
    counter = Counter(answers)
    majority_answer, count = counter.most_common(1)[0]

    return {
        "majority_answer": majority_answer,
        "confidence": count / n_samples,
        "all_answers": dict(counter),
        "all_responses": full_responses
    }


# Test Self-Consistency
sc_prompt = """
Một AI startup có 3 engineers. Mỗi engineer build 2 AI agents.
Mỗi agent gọi trung bình 5 API calls/request.
Nếu system nhận 100 requests/ngày, tổng API calls trong 1 tuần là bao nhiêu?

Suy nghĩ từng bước và đưa ra kết quả cuối cùng là một con số.
"""

result = self_consistency(sc_prompt, n_samples=5)
print(f"\n📊 KẾT QUẢ SELF-CONSISTENCY:")
print(f"   Majority answer : {result['majority_answer']}")
print(f"   Confidence      : {result['confidence']:.0%}")
print(f"   Tất cả đáp án   : {result['all_answers']}")

In [ ]:
# ============================================================
# 3.4 TREE-OF-THOUGHT (ToT)
# Khám phá nhiều reasoning branches song song, đánh giá và chọn tốt nhất
# ============================================================

tot_prompt = """
Problem: Thiết kế kiến trúc memory cho một AI Agent phải xử lý 10,000 sessions/ngày,
mỗi session có 50 turns conversation.

Đề xuất 3 approaches KHÁC NHAU:

Approach 1: [Tên kiến trúc]
→ Mô tả: ...
→ Pros: ...
→ Cons: ...
→ Score: .../10

Approach 2: [Tên kiến trúc]
→ Mô tả: ...
→ Pros: ...
→ Cons: ...
→ Score: .../10

Approach 3: [Tên kiến trúc]
→ Mô tả: ...
→ Pros: ...
→ Cons: ...
→ Score: .../10

Final Recommendation: [Approach được chọn và lý do]
"""

print("🌳 TREE-OF-THOUGHT:")
_ = chat(tot_prompt, model=SMART_MODEL, max_tokens=1500)

### 🎯 Khi nào dùng kỹ thuật nào?

| Kỹ thuật | Khi nào dùng | Trade-off |
|----------|-------------|----------|
| **Zero-shot CoT** | Bài toán reasoning đơn giản, 1 đáp án rõ ràng | Nhanh, rẻ |
| **Few-shot CoT** | Domain-specific reasoning, cần dạy reasoning pattern cụ thể | Tốn token hơn |
| **Self-Consistency** | Yêu cầu accuracy cao, chấp nhận latency & cost × n | n× chi phí |
| **Tree-of-Thought** | Tìm giải pháp tốt nhất trong không gian tìm kiếm lớn | Phức tạp, tốn kém nhất |

---
## 👤 Phần 4: Role Prompting & System Prompt Design

In [ ]:
# ============================================================
# 4.1 So sánh: Vague Role vs Specific Role
# ============================================================

code_to_review = """
async def get_user(user_id: int, db):
    user = db.query("SELECT * FROM users WHERE id = " + str(user_id)).first()
    return user
"""

# ❌ Role mơ hồ
bad_system = "You are a helpful AI assistant."

# ✅ Role cụ thể với context, style, và priorities
good_system = """
You are a Senior Python Engineer with 10+ years building production backend systems at scale.
You specialize in security vulnerabilities and database performance.

When reviewing code, you:
1. ALWAYS identify security vulnerabilities first (SQL injection, XSS, auth issues)
2. Then flag performance bottlenecks with specific metrics
3. Provide corrected code, not just descriptions
4. Explain WHY each issue matters in production
5. Rate severity: CRITICAL / HIGH / MEDIUM / LOW
"""

user_msg = f"Review this code:\n```python\n{code_to_review}\n```"

print("❌ BAD (vague role):")
_ = chat(user_msg, system_message=bad_system, max_tokens=300)

print("\n" + "=" * 60)
print("✅ GOOD (specific role):")
_ = chat(user_msg, system_message=good_system, max_tokens=500)

In [ ]:
# ============================================================
# 4.2 ANATOMY OF A PROFESSIONAL SYSTEM PROMPT
# Template đầy đủ cho một AI Agent production
# ============================================================

CODE_REVIEW_AGENT_SYSTEM_PROMPT = """
# IDENTITY
You are CodeReview Agent, an expert Python code reviewer with 15 years of experience.
You have reviewed 50,000+ pull requests at FAANG-scale companies.

# OBJECTIVE
Task: Analyze provided code and deliver actionable, specific feedback.
Audience: Experienced Python developers — skip basic explanations.

# CAPABILITIES
You can identify:
- Security vulnerabilities (SQL injection, XSS, SSRF, auth bypass, etc.)
- Performance bottlenecks (N+1 queries, missing indexes, memory leaks)
- Code quality issues (naming, coupling, testability)
- Production readiness gaps (error handling, logging, monitoring)

# CONSTRAINTS
- Do NOT auto-fix code unless explicitly asked
- Do NOT comment on style unless it affects readability significantly
- Maximum 8 issues per review (prioritize by severity)
- If no issues found, explicitly state: "Approved ✅"

# OUTPUT FORMAT
Use exactly this structure:

## 📋 Summary
[1–2 sentence overview]

## 🚨 Critical Issues (must fix before merge)
- [ISSUE_TYPE] Description → Impact → Fix suggestion

## ⚠️ Suggestions (should fix)
- Description → Why it matters

## ✅ Positives
- What was done well

# BEHAVIOR RULES
- Missing tests → Always list under Critical Issues
- Security issue → ALWAYS critical, never downgrade
- Ambiguous code → Ask for clarification, don't assume intent
- Priority: security > correctness > performance > style
"""

# Test với code có nhiều vấn đề
vulnerable_code = """
import sqlite3
import pickle

def get_user_data(username, password):
    conn = sqlite3.connect('app.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    cursor.execute(query)
    result = cursor.fetchall()
    conn.close()
    return result

def load_user_preferences(user_id):
    with open(f'/tmp/prefs_{user_id}.pkl', 'rb') as f:
        return pickle.load(f)  # Load user preferences
"""

print("🔍 CODE REVIEW AGENT:")
_ = chat(
    f"Review this code:\n```python\n{vulnerable_code}\n```",
    system_message=CODE_REVIEW_AGENT_SYSTEM_PROMPT,
    model=SMART_MODEL,
    max_tokens=1000
)

In [ ]:
# ============================================================
# 4.3 MULTI-PERSONA cho Multi-Agent Systems
# Mỗi agent cần persona riêng biệt để tránh role confusion
# ============================================================

ORCHESTRATOR_PROMPT = """
You are the Orchestrator Agent.

Your ONLY responsibilities:
1. Receive the user's task
2. Decompose it into clear subtasks
3. Specify which specialist agent handles each subtask
4. Synthesize results into a final response

You do NOT execute subtasks yourself. You ONLY coordinate.

Output format:
{
  "task_analysis": "...",
  "subtasks": [
    {"id": 1, "agent": "researcher|coder|reviewer", "task": "...", "depends_on": []}
  ],
  "execution_order": [1, 2, 3]
}
"""

RESEARCHER_PROMPT = """
You are the Research Agent.
ONLY search for and return information when requested by the Orchestrator.
Return raw information — do NOT interpret, recommend, or add opinions.
Format: JSON with fields: source, content, confidence_score (0–1)
"""

CODER_PROMPT = """
You are the Code Generation Agent.
ONLY write code when given a clear specification.
Always include: type hints, docstring, error handling, and at least 1 usage example.
Return only the code block — no explanations unless asked.
"""

# Simulate Orchestrator decomposing a task
user_task = """
Build a function that fetches real-time stock prices from an API
and stores them in a SQLite database with proper error handling.
"""

print("🎭 ORCHESTRATOR AGENT:")
orchestrator_result = chat(
    f"Task from user: {user_task}",
    system_message=ORCHESTRATOR_PROMPT,
    model=SMART_MODEL,
    max_tokens=600
)

---
## 📊 Phần 5: Structured Output & JSON Mode

In [ ]:
# ============================================================
# 5.1 JSON MODE của OpenAI
# response_format={"type": "json_object"} đảm bảo output luôn là valid JSON
# ============================================================

def get_json_output(prompt: str, schema_description: str) -> dict:
    """
    Luôn trả về valid JSON.
    Quan trọng: Phải mention "JSON" trong system hoặc user message khi dùng json_object mode.
    """
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        temperature=0.1,
        max_tokens=1000,
        response_format={"type": "json_object"},  # ← Key parameter!
        messages=[
            {
                "role": "system",
                "content": f"""
You are a data extraction agent.
ALWAYS respond with valid JSON following this schema:
{schema_description}
Rules:
- Output JSON only, no additional text
- Use null for missing data, never omit fields
"""
            },
            {"role": "user", "content": prompt}
        ]
    )
    return json.loads(response.choices[0].message.content)


# Test: Extract structured info từ unstructured text
job_posting = """
Chúng tôi cần tuyển Senior AI Engineer với 5+ năm kinh nghiệm Python.
Yêu cầu: FastAPI, LangChain, Docker, kinh nghiệm deploy ML models.
Mức lương: 3000–5000 USD/tháng. Remote-first. Deadline nộp hồ sơ: 31/12/2024.
Email: career@aicompany.com
"""

schema = """
{
  "position": "string",
  "seniority": "junior|mid|senior|lead",
  "years_experience": "number or null",
  "required_skills": ["string"],
  "salary_range": {"min": "number or null", "max": "number or null", "currency": "string"},
  "work_mode": "remote|hybrid|onsite",
  "deadline": "YYYY-MM-DD or null",
  "contact": "string or null"
}
"""

result = get_json_output(f"Extract info from: {job_posting}", schema)
print("📊 Structured Output:")
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 5.2 PYDANTIC VALIDATION cho structured outputs
# Đây là pattern production-grade nhất
# ============================================================

from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class Severity(str, Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class CodeIssue(BaseModel):
    type: str = Field(description="Issue type: SECURITY, PERFORMANCE, CORRECTNESS, STYLE")
    severity: Severity
    description: str
    line_number: Optional[int] = None
    fix_suggestion: str


class CodeReviewResult(BaseModel):
    summary: str
    approved: bool
    overall_score: int = Field(ge=0, le=10, description="0-10 score")
    issues: List[CodeIssue]
    positives: List[str]

    @field_validator("overall_score")
    @classmethod
    def validate_score(cls, v):
        if not 0 <= v <= 10:
            raise ValueError("Score must be 0-10")
        return v


def review_code_structured(code: str) -> CodeReviewResult:
    schema_json = CodeReviewResult.model_json_schema()

    response = client.chat.completions.create(
        model=SMART_MODEL,
        temperature=0.1,
        max_tokens=1500,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": f"""You are an expert code reviewer.
Return JSON matching this schema exactly:
{json.dumps(schema_json, indent=2)}"""
            },
            {"role": "user", "content": f"Review this Python code:\n```python\n{code}\n```"}
        ]
    )

    raw = json.loads(response.choices[0].message.content)
    return CodeReviewResult(**raw)  # Pydantic validation!


# Test
sample_code = """
def divide(a, b):
    return a / b  # No error handling

result = divide(10, 0)  # Will crash!
"""

review = review_code_structured(sample_code)
print(f"✅ Pydantic validated output:")
print(f"   Approved    : {review.approved}")
print(f"   Score       : {review.overall_score}/10")
print(f"   Issues      : {len(review.issues)}")
print(f"   Summary     : {review.summary}")
if review.issues:
    print(f"\n   Issues found:")
    for issue in review.issues:
        print(f"   [{issue.severity}] {issue.type}: {issue.description}")
        print(f"   → Fix: {issue.fix_suggestion}")

---
## 📋 Phần 6: Prompt Templates & Chaining

In [ ]:
# ============================================================
# 6.1 JINJA2 TEMPLATES cho Production Prompts
# Trong production, prompts KHÔNG BAO GIỜ hardcoded
# ============================================================

from jinja2 import Template

AGENT_TEMPLATE = Template("""
# ROLE
{{ role }}

# TASK
{{ task }}

# CONTEXT
{{ context }}

{% if constraints %}
# CONSTRAINTS
{% for constraint in constraints %}
- {{ constraint }}
{% endfor %}
{% endif %}

{% if examples %}
# EXAMPLES
{% for ex in examples %}
Input: {{ ex.input }}
Output: {{ ex.output }}
---
{% endfor %}
{% endif %}

# OUTPUT FORMAT
{{ output_format }}
""")


@dataclass
class PromptConfig:
    role: str
    task: str
    context: str
    output_format: str
    constraints: list = field(default_factory=list)
    examples: list = field(default_factory=list)


# Build prompt from template
config = PromptConfig(
    role="Senior Python Code Reviewer with security expertise",
    task="Review the provided Python code for bugs and security issues",
    context="Production FastAPI service, 10,000 requests/minute, Python 3.12",
    output_format='JSON: {"bugs": [...], "security_issues": [...], "score": 0-10}',
    constraints=[
        "Do not suggest a complete rewrite",
        "Max 5 issues",
        "Focus on security and correctness over style"
    ]
)

rendered_prompt = AGENT_TEMPLATE.render(**config.__dict__)
print("📝 Rendered System Prompt:")
print(rendered_prompt)
print(f"\n📊 Token count: {count_tokens(rendered_prompt)} tokens")

In [ ]:
# ============================================================
# 6.2 PROMPT CHAINING PATTERN
# Chia task phức tạp thành nhiều steps, output step i → input step i+1
# ============================================================

def analyze_codebase_with_chaining(code: str) -> dict:
    """
    3-step prompt chain: Extract → Analyze → Recommend
    Tại sao chain thay vì 1 prompt to?
    - Mỗi step chuyên biệt → accuracy cao hơn
    - Dễ debug: biết step nào fail
    - Có thể cache intermediate results
    - Models xử lý tốt hơn khi task nhỏ hơn
    """
    print("🔗 PROMPT CHAIN - 3 STEPS")
    print("=" * 60)

    # STEP 1: Extract structure
    print("\n📍 STEP 1: Extracting code structure...")
    step1_response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        max_tokens=800,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You are a code parser. Extract the structure as JSON: {functions, classes, imports, dependencies}"
            },
            {"role": "user", "content": f"Parse this Python code:\n{code}"}
        ]
    )
    structure = step1_response.choices[0].message.content
    print(f"   Structure extracted: {len(structure)} chars")

    # STEP 2: Security analysis
    print("\n📍 STEP 2: Security analysis...")
    step2_response = client.chat.completions.create(
        model=SMART_MODEL,
        max_tokens=800,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "You are a security expert. Analyze the code structure for vulnerabilities. Return JSON: {vulnerabilities: [{type, severity, description}]}"
            },
            {"role": "user", "content": f"Code structure:\n{structure}\n\nOriginal code:\n{code}"}
        ]
    )
    analysis = step2_response.choices[0].message.content
    print(f"   Analysis complete: {len(analysis)} chars")

    # STEP 3: Recommendations
    print("\n📍 STEP 3: Generating recommendations...")
    step3_response = client.chat.completions.create(
        model=SMART_MODEL,
        max_tokens=1000,
        messages=[
            {
                "role": "system",
                "content": "You are a senior architect. Based on the security analysis, propose a concrete, prioritized action plan with code examples."
            },
            {
                "role": "user",
                "content": f"Security analysis:\n{analysis}\n\nOriginal code:\n{code}\n\nGenerate action plan with fixed code."
            }
        ]
    )
    plan = step3_response.choices[0].message.content

    print("\n📊 FINAL RESULT:")
    print("-" * 40)
    print(plan)

    return {
        "structure": json.loads(structure),
        "analysis": json.loads(analysis),
        "action_plan": plan
    }


# Test
insecure_code = """
import os
import sqlite3

def execute_command(cmd: str):
    result = os.system(cmd)  # Direct OS command execution!
    return result

def search_user(username: str, db_path: str):
    conn = sqlite3.connect(db_path)
    # SQL Injection vulnerability
    rows = conn.execute(f"SELECT * FROM users WHERE name = '{username}'").fetchall()
    return rows
"""

chain_result = analyze_codebase_with_chaining(insecure_code)

---
## 🔄 Phần 7: ReAct Pattern (Reason + Act)

In [ ]:
# ============================================================
# 7. ReAct PATTERN — Backbone của AI Agent Frameworks
# Thought → Action → Observation → Thought → Action → ...
# ============================================================

# Define mock tools (trong production: gọi API thật)
def search_web(query: str) -> str:
    """Mock: Simulate web search."""
    mock_results = {
        "gpt-4o pricing": "GPT-4o: $2.50/1M input tokens, $10.00/1M output tokens (as of 2024)",
        "openai rate limits": "GPT-4o: 10,000 RPM, 30M TPM for Tier 5 accounts",
        "python asyncio": "asyncio is Python's built-in library for async I/O, ideal for concurrent LLM calls",
    }
    for key, val in mock_results.items():
        if key.lower() in query.lower():
            return val
    return f"Search results for '{query}': Found 847 results. Top result: [relevant info about {query}]"


def calculate(expression: str) -> str:
    """Evaluate a math expression safely."""
    try:
        # Safety: chỉ cho phép math operations
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def read_file(path: str) -> str:
    """Mock: Read a file."""
    return f"Content of {path}: [Mock file content with 2500 tokens of Python code]"


# Tool registry
TOOLS = {
    "search": search_web,
    "calculate": calculate,
    "read_file": read_file
}


def parse_react_action(text: str) -> tuple[Optional[str], Optional[str]]:
    """Parse Action: tool_name(params) from ReAct output."""
    import re
    action_match = re.search(r'Action:\s*(\w+)\((.*)\)', text)
    if action_match:
        tool_name = action_match.group(1).strip()
        tool_input = action_match.group(2).strip().strip('"\'')
        return tool_name, tool_input
    return None, None


REACT_SYSTEM_PROMPT = """
Solve tasks using this EXACT loop:

Thought: [Your reasoning about what to do next]
Action: [tool_name]("[parameters]")
Observation: [Result returned by the tool]
... (repeat as needed)
Thought: [Final reasoning with all info gathered]
Final Answer: [Complete answer to the user's question]

Available tools:
- search(query: str): Search the web for information
- calculate(expression: str): Evaluate a math expression
- read_file(path: str): Read a file from disk

IMPORTANT: Always start with Thought, then Action OR Final Answer.
If you have enough info, go straight to Final Answer.
"""


def run_react_agent(user_query: str, max_steps: int = 6) -> str:
    """
    Simple ReAct agent loop.
    Trong production: dùng LangGraph hoặc LangChain agents.
    """
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    print(f"🤖 REACT AGENT running...")
    print(f"Query: {user_query}")
    print("=" * 60)

    for step in range(max_steps):
        print(f"\n--- Step {step + 1} ---")

        response = client.chat.completions.create(
            model=SMART_MODEL,
            temperature=0.1,
            max_tokens=500,
            messages=messages
        )

        agent_output = response.choices[0].message.content
        print(agent_output)

        # Check if done
        if "Final Answer:" in agent_output:
            print("\n✅ Agent completed!")
            return agent_output.split("Final Answer:")[-1].strip()

        # Parse and execute action
        tool_name, tool_input = parse_react_action(agent_output)
        if tool_name and tool_name in TOOLS:
            observation = TOOLS[tool_name](tool_input)
            print(f"\nObservation: {observation}")

            # Add to conversation history
            messages.append({"role": "assistant", "content": agent_output})
            messages.append({
                "role": "user",
                "content": f"Observation: {observation}\n\nContinue with the next Thought:"
            })
        else:
            print("⚠️ No valid action found, stopping.")
            break

    return "Max steps reached."


# Test ReAct Agent
answer = run_react_agent(
    "What is the total monthly cost of running 500,000 GPT-4o API calls per day, "
    "assuming 1000 input tokens and 500 output tokens per call?"
)

---
## 🔬 Phần 8: Advanced Techniques

In [ ]:
# ============================================================
# 8.1 META-PROMPTING — Dùng LLM để generate và optimize prompts
# ============================================================

meta_prompt_template = """
You are an expert Prompt Engineer with deep knowledge of GPT-4o capabilities.

Generate an OPTIMAL system prompt for this use case:

Task Description: {task_description}
Target Model: GPT-4o
Required Output Format: {output_spec}
Constraints: {constraints}
Audience: {audience}

Generate a complete system prompt that includes:
1. Clear role definition with specific expertise
2. Detailed task instructions
3. 2 few-shot examples relevant to the task
4. Explicit output format specification
5. Edge case handling rules

After the prompt, explain your design decisions in 3-5 bullet points.
"""

use_case = {
    "task_description": "Classify customer support tickets by urgency and route to correct team",
    "output_spec": "JSON: {ticket_id, urgency: low|medium|high|critical, team: engineering|billing|sales|legal, reason, estimated_response_hours}",
    "constraints": "Must respond in < 500ms; no hallucinated routing; always justify routing decision",
    "audience": "Customer support automation system processing 5000 tickets/day"
}

generated_prompt = meta_prompt_template.format(**use_case)

print("🔮 META-PROMPTING — Generating optimal prompt:")
_ = chat(generated_prompt, model=SMART_MODEL, max_tokens=1200)

In [ ]:
# ============================================================
# 8.2 SELF-CRITIQUE & REFLECTION
# Dạy model đánh giá và cải thiện output của chính mình
# ============================================================

def self_critique_pipeline(task: str, criteria: list[str]) -> dict:
    """
    3-step reflection pipeline:
    1. Generate initial response
    2. Critique against criteria
    3. Generate improved version
    """
    print("🔄 SELF-CRITIQUE PIPELINE")
    print("=" * 60)

    # Step 1: Initial response
    print("\n📍 Step 1: Generating initial response...")
    initial = client.chat.completions.create(
        model=DEFAULT_MODEL,
        max_tokens=600,
        messages=[{"role": "user", "content": task}]
    ).choices[0].message.content
    print(initial)

    # Step 2: Self-critique
    print("\n" + "=" * 60)
    print("\n📍 Step 2: Critiquing...")
    criteria_text = "\n".join([f"- {c}" for c in criteria])
    critique_prompt = f"""
Here is a response to the task: "{task}"

RESPONSE:
{initial}

Critically evaluate this response against these criteria:
{criteria_text}

For each criterion, rate: ✅ Pass / ⚠️ Partial / ❌ Fail
Then provide specific improvements needed.
"""
    critique = client.chat.completions.create(
        model=SMART_MODEL,
        max_tokens=600,
        messages=[{"role": "user", "content": critique_prompt}]
    ).choices[0].message.content
    print(critique)

    # Step 3: Improved response
    print("\n" + "=" * 60)
    print("\n📍 Step 3: Generating improved response...")
    improve_prompt = f"""
Task: {task}

Initial response:
{initial}

Critique and issues identified:
{critique}

Now write a COMPLETE, IMPROVED response that addresses all critique points.
Do not reference the critique — just write the better answer directly.
"""
    improved = client.chat.completions.create(
        model=SMART_MODEL,
        max_tokens=800,
        messages=[{"role": "user", "content": improve_prompt}]
    ).choices[0].message.content
    print(improved)

    return {"initial": initial, "critique": critique, "improved": improved}


# Test
result = self_critique_pipeline(
    task="Explain what an AI Agent is and when to use one vs a simple LLM call",
    criteria=[
        "Accuracy: Technically correct definition",
        "Completeness: Covers both what it IS and WHEN to use",
        "Clarity: A junior developer can understand",
        "Actionability: Provides concrete decision criteria",
        "Conciseness: Under 200 words"
    ]
)

---
## 📈 Phần 9: Prompt Evaluation & A/B Testing

In [ ]:
# ============================================================
# 9. PROMPT EVALUATION FRAMEWORK
# Professional PE đòi hỏi evaluation framework, không phải cảm tính
# ============================================================

@dataclass
class EvalCase:
    input: str
    expected_output: str
    weight: float = 1.0
    description: str = ""


class PromptEvaluator:
    def __init__(self, eval_cases: list[EvalCase]):
        self.cases = eval_cases

    def llm_scorer(self, actual: str, expected: str) -> float:
        """
        LLM-as-judge: Dùng GPT để đánh giá similarity.
        Đây là cách phổ biến nhất trong production (OpenAI Evals, LangSmith).
        """
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            temperature=0,
            max_tokens=100,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "You are an evaluation judge. Return JSON: {score: 0.0-1.0, reason: string}"
                },
                {
                    "role": "user",
                    "content": f"""Compare actual vs expected output.
Score 1.0 = perfect match in meaning and format.
Score 0.5 = correct meaning but wrong format.
Score 0.0 = wrong meaning.

Expected: {expected}
Actual: {actual}"""
                }
            ]
        )
        result = json.loads(response.choices[0].message.content)
        return float(result["score"])

    def evaluate_prompt(self, system_prompt: str, n_runs: int = 1) -> dict:
        """Evaluate a system prompt across all test cases."""
        scores = []
        latencies = []
        token_costs = []

        for case in self.cases:
            run_scores = []
            for _ in range(n_runs):
                start = time.time()
                response = client.chat.completions.create(
                    model=DEFAULT_MODEL,
                    temperature=0.1,
                    max_tokens=300,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": case.input}
                    ]
                )
                latency = time.time() - start
                actual = response.choices[0].message.content
                score = self.llm_scorer(actual, case.expected_output) * case.weight
                run_scores.append(score)
                latencies.append(latency)
                token_costs.append(response.usage.total_tokens)

            scores.append(statistics.mean(run_scores))

        return {
            "mean_score":   round(statistics.mean(scores), 3),
            "min_score":    round(min(scores), 3),
            "pass_rate":    round(sum(1 for s in scores if s >= 0.7) / len(scores), 3),
            "avg_latency":  round(statistics.mean(latencies), 2),
            "avg_tokens":   round(statistics.mean(token_costs)),
        }


# Define test cases for sentiment analysis
eval_cases = [
    EvalCase(
        input="Review: Great product, fast delivery!",
        expected_output="POSITIVE",
        weight=1.0,
        description="Clear positive"
    ),
    EvalCase(
        input="Review: Product is okay but could be better",
        expected_output="NEUTRAL",
        weight=1.0,
        description="Mixed/neutral"
    ),
    EvalCase(
        input="Review: Terrible quality, broken on arrival, no refund",
        expected_output="NEGATIVE",
        weight=1.5,  # Edge cases weighted higher
        description="Clear negative with frustration"
    ),
]

# A/B Test: Compare two system prompts
prompt_a = "Classify sentiment as POSITIVE, NEGATIVE, or NEUTRAL."

prompt_b = """
You are a sentiment analysis expert for e-commerce.
Classify customer reviews as exactly one of: POSITIVE, NEGATIVE, NEUTRAL.

Rules:
- POSITIVE: Overall satisfied, recommends, positive experience
- NEGATIVE: Dissatisfied, problems, would not recommend
- NEUTRAL: Mixed feelings, neither clearly positive nor negative

Output only the label, nothing else.
"""

evaluator = PromptEvaluator(eval_cases)

print("📊 A/B TEST: Evaluating prompts...")
print("\n🅰️ Evaluating Prompt A (simple)...")
result_a = evaluator.evaluate_prompt(prompt_a)

print("\n🅱️ Evaluating Prompt B (detailed)...")
result_b = evaluator.evaluate_prompt(prompt_b)

print("\n" + "=" * 60)
print("📈 A/B TEST RESULTS:")
print(f"{'Metric':<20} {'Prompt A':>12} {'Prompt B':>12} {'Winner':>8}")
print("-" * 55)
for key in ["mean_score", "min_score", "pass_rate", "avg_latency", "avg_tokens"]:
    a_val = result_a[key]
    b_val = result_b[key]
    # Lower is better for latency/tokens, higher for scores
    if key in ["avg_latency", "avg_tokens"]:
        winner = "A ✅" if a_val < b_val else "B ✅"
    else:
        winner = "A ✅" if a_val > b_val else "B ✅"
    print(f"{key:<20} {str(a_val):>12} {str(b_val):>12} {winner:>8}")

---
## 🏭 Phần 10: Production — Smart Prompt Manager

In [ ]:
# ============================================================
# 10. SMART PROMPT MANAGER
# Production-grade: versioning, caching, token counting, cost tracking
# ============================================================

from pydantic import BaseModel


class PromptVersion(BaseModel):
    name: str
    version: str
    system_template: str
    user_template: str
    model: str = DEFAULT_MODEL
    max_tokens: int = 1000
    temperature: float = 0.1
    variables: list[str] = []


class UsageStats(BaseModel):
    total_calls: int = 0
    cache_hits: int = 0
    total_tokens: int = 0
    total_cost_usd: float = 0.0


class SmartPromptManager:
    """
    Production-grade prompt manager với:
    - Version control cho prompts
    - Response caching (giảm cost)
    - Token counting và cost tracking
    - Usage analytics
    """

    # GPT-4o-mini pricing (per 1M tokens)
    PRICING = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o":      {"input": 2.50, "output": 10.00},
    }

    def __init__(self):
        self.prompts: dict[str, PromptVersion] = {}
        self._cache: dict[str, str] = {}
        self.stats: dict[str, UsageStats] = {}

    def register(self, prompt: PromptVersion):
        key = f"{prompt.name}:{prompt.version}"
        self.prompts[key] = prompt
        self.stats[key] = UsageStats()
        print(f"✅ Registered prompt: {key}")

    def _render(self, template: str, **kwargs) -> str:
        for k, v in kwargs.items():
            template = template.replace(f"{{{k}}}", str(v))
        return template

    def _cache_key(self, prompt_key: str, **kwargs) -> str:
        content = f"{prompt_key}:{json.dumps(kwargs, sort_keys=True)}"
        return hashlib.md5(content.encode()).hexdigest()

    def _calculate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        if model not in self.PRICING:
            return 0.0
        pricing = self.PRICING[model]
        return (input_tokens * pricing["input"] + output_tokens * pricing["output"]) / 1_000_000

    def run(
        self,
        prompt_name: str,
        version: str = "v1",
        use_cache: bool = True,
        **kwargs
    ) -> str:
        key = f"{prompt_name}:{version}"
        if key not in self.prompts:
            raise ValueError(f"Prompt '{key}' not found. Register it first.")

        pv = self.prompts[key]
        stats = self.stats[key]
        stats.total_calls += 1

        # Check cache
        cache_key = self._cache_key(key, **kwargs)
        if use_cache and cache_key in self._cache:
            stats.cache_hits += 1
            print(f"⚡ Cache hit! (saved ~${self._calculate_cost(pv.model, 500, 200):.6f})")
            return self._cache[cache_key]

        # Render templates
        system = self._render(pv.system_template, **kwargs)
        user = self._render(pv.user_template, **kwargs)

        # Count input tokens
        input_tokens = count_tokens(system + user)

        # API call
        response = client.chat.completions.create(
            model=pv.model,
            temperature=pv.temperature,
            max_tokens=pv.max_tokens,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user}
            ]
        )

        result = response.choices[0].message.content
        output_tokens = response.usage.completion_tokens
        cost = self._calculate_cost(pv.model, response.usage.prompt_tokens, output_tokens)

        # Update stats
        stats.total_tokens += response.usage.total_tokens
        stats.total_cost_usd += cost

        # Cache result
        if use_cache:
            self._cache[cache_key] = result

        print(f"💰 Cost: ${cost:.6f} | Tokens: {response.usage.total_tokens}")
        return result

    def get_stats(self, prompt_name: str, version: str = "v1") -> dict:
        key = f"{prompt_name}:{version}"
        stats = self.stats.get(key, UsageStats())
        cache_rate = stats.cache_hits / stats.total_calls if stats.total_calls > 0 else 0
        return {
            "prompt":       key,
            "total_calls":  stats.total_calls,
            "cache_hits":   stats.cache_hits,
            "cache_rate":   f"{cache_rate:.0%}",
            "total_tokens": stats.total_tokens,
            "total_cost":   f"${stats.total_cost_usd:.6f}",
        }


# ============================================================
# Demo: Register và sử dụng Smart Prompt Manager
# ============================================================

manager = SmartPromptManager()

# Register prompt v1
manager.register(PromptVersion(
    name="sentiment_analyzer",
    version="v1",
    system_template="You are a sentiment analysis expert for {domain}.",
    user_template="Classify sentiment (POSITIVE/NEGATIVE/NEUTRAL) of: {text}. Reply with one word.",
    model=DEFAULT_MODEL,
    temperature=0.0,
    variables=["domain", "text"]
))

# Register improved v2
manager.register(PromptVersion(
    name="sentiment_analyzer",
    version="v2",
    system_template="""You are a senior sentiment analysis expert for {domain}.
Reply with exactly one word: POSITIVE, NEGATIVE, or NEUTRAL.
POSITIVE = overall satisfied. NEGATIVE = dissatisfied. NEUTRAL = mixed.""",
    user_template="Classify: {text}",
    model=DEFAULT_MODEL,
    temperature=0.0,
    variables=["domain", "text"]
))

# Use the manager
reviews = [
    "Hàng đẹp, giao nhanh, shop nhiệt tình!",
    "Sản phẩm bình thường, không quá tốt không quá tệ",
    "Hàng kém chất lượng, thất vọng hoàn toàn",
    "Hàng đẹp, giao nhanh, shop nhiệt tình!",  # Duplicate → cache hit!
]

print("\n🚀 RUNNING SMART PROMPT MANAGER:\n")
for review in reviews:
    print(f"\nReview: {review}")
    result = manager.run(
        "sentiment_analyzer", version="v2",
        domain="e-commerce", text=review
    )
    print(f"→ Sentiment: {result}")

print("\n" + "=" * 60)
print("📊 USAGE STATS:")
stats = manager.get_stats("sentiment_analyzer", "v2")
for k, v in stats.items():
    print(f"   {k:<15}: {v}")

---
## 🚫 Phần Bonus: Common Anti-Patterns

In [ ]:
# ============================================================
# ANTI-PATTERNS — Những lỗi phổ biến nhất cần tránh
# ============================================================

print("""
🚫 TOP 6 PROMPT ENGINEERING ANTI-PATTERNS
==========================================

1. ❌ Vague Instructions
   BAD:  "Be helpful and answer well"
   GOOD: "Answer in bullet points, max 5 points, technical audience, Python 3.12 context"

2. ❌ Negative-only Constraints
   BAD:  "Don't be verbose. Don't use jargon."
   GOOD: "Write concisely (max 3 sentences). Use simple English, no acronyms."
   WHY:  Models still violate negative constraints ~30% of time

3. ❌ Overloaded Single Prompts
   BAD:  "Summarize + translate + format + analyze + recommend" in 1 prompt
   GOOD: Break into prompt chain (Section 6.2)
   WHY:  Models handle earlier tasks in a list better than later ones

4. ❌ Inconsistent Few-shot Examples
   BAD:  Example 1: JSON output, Example 2: markdown output, Example 3: plain text
   GOOD: All examples use identical structure
   WHY:  Contradictory examples → unstable, unpredictable outputs

5. ❌ Ignoring Model-specific Quirks
   - GPT-4o: Use function calling for tools, not text-based ReAct
   - Claude: XML tags work much better than markdown sections
   - Llama/Mistral: Embed instructions in user turn, not system
   WHY:  Same prompt can have 40% performance difference across models

6. ❌ No Output Constraints
   BAD:  "Analyze this code"
   GOOD: "Analyze this code. Return exactly 3 bullet points. Each under 20 words."
   WHY:  Unconstrained output wastes tokens and is harder to parse
""")

# Demo: Anti-pattern #1 fix
print("\n🧪 DEMO: Vague vs Specific Prompt")
print("=" * 60)

topic = "What is a vector database?"

print("❌ Vague:")
_ = chat(topic, system_message="Be helpful.")

print("\n✅ Specific:")
_ = chat(
    topic,
    system_message="""Explain technical concepts to senior Python developers.
Format: 1 sentence definition + 3 bullet points on key properties + 1 line on when to use.
Total: max 100 words. Include one concrete example."""
)

---
## 🎯 Bài Tập Thực Hành

Hoàn thành các bài tập sau để củng cố kiến thức:

In [ ]:
# ============================================================
# 💪 BÀI TẬP 1: Few-shot Classification
# Xây dựng few-shot prompt để phân loại bug reports
# ============================================================

# TODO: Hoàn thiện prompt này với ít nhất 3 examples
bug_classification_prompt = """
# Phân loại bug reports theo loại:
# - UI_BUG: Vấn đề về giao diện, hiển thị
# - PERFORMANCE: Chậm, lag, tốn bộ nhớ
# - LOGIC_ERROR: Kết quả sai, tính toán sai
# - SECURITY: Vấn đề bảo mật
# - DATA_LOSS: Mất dữ liệu

# TODO: Thêm ít nhất 3 examples ở đây
# Example 1:
# Bug: "..."
# Type: ...

# Now classify:
Bug: "Sau khi update, button Submit không hiển thị trên mobile Safari"
Type:
"""

print("💪 Bài tập 1: Few-shot Bug Classification")
print("TODO: Hoàn thiện prompt ở trên và uncomment dòng sau:")
# _ = chat(bug_classification_prompt)
print("[Uncomment dòng trên sau khi thêm examples vào prompt]")

In [ ]:
# ============================================================
# 💪 BÀI TẬP 2: Build Your Own ReAct Agent
# Thêm 2 tools mới vào ReAct Agent và test
# ============================================================

# TODO: Implement 2 tools mới
def get_current_time(timezone: str = "UTC") -> str:
    """TODO: Implement this tool - trả về current time theo timezone"""
    import datetime
    return f"Current time in {timezone}: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} (mock)"


def convert_currency(amount_and_currency: str) -> str:
    """TODO: Implement this tool - convert currency
    Input format: '100 USD to VND'
    """
    # Mock rates
    return f"Converting {amount_and_currency}: [TODO: implement with real rates]"


# TODO: Thêm 2 tools trên vào TOOLS dictionary
TOOLS_EXTENDED = {
    **TOOLS,
    # "get_time": get_current_time,
    # "convert_currency": convert_currency,
}

print("💪 Bài tập 2: Implement và test 2 tools mới")
print("TODO:")
print("  1. Complete get_current_time() implementation")
print("  2. Complete convert_currency() implementation")
print("  3. Add them to TOOLS_EXTENDED")
print("  4. Update REACT_SYSTEM_PROMPT với tools mới")
print("  5. Test với query: 'How much is 500 USD in VND right now?'")

In [ ]:
# ============================================================
# 💪 BÀI TẬP 3: Prompt A/B Testing
# So sánh 3 versions của cùng một prompt
# ============================================================

# TODO: Viết 3 versions khác nhau của prompt dưới đây
# Task: Tóm tắt một đoạn code thành 1 câu mô tả chức năng

# Version A: Minimal prompt
prompt_version_a = "Summarize what this code does in one sentence."

# Version B: Role-based prompt  
prompt_version_b = """# TODO: Viết prompt với role prompting"""

# Version C: Few-shot prompt
prompt_version_c = """# TODO: Viết prompt với 2-3 examples"""

test_code = """
def fibonacci(n: int) -> list[int]:
    if n <= 0:
        return []
    if n == 1:
        return [0]
    sequence = [0, 1]
    while len(sequence) < n:
        sequence.append(sequence[-1] + sequence[-2])
    return sequence
"""

print("💪 Bài tập 3: A/B/C Test 3 prompt versions")
print("\nVersion A (baseline):")
_ = chat(f"{prompt_version_a}\n\nCode:\n{test_code}")

print("\nTODO: Implement và test Version B và C")
print("Đánh giá: Phiên bản nào cho kết quả tốt nhất? Tại sao?")

---
## 📋 Tổng Kết & Roadmap Tiếp Theo

### ✅ Bạn đã học được:

| Kỹ thuật | Status |
|----------|--------|
| Zero-shot, One-shot, Few-shot Prompting | ✅ |
| Dynamic Few-shot Selection | ✅ |
| Chain-of-Thought (Zero-shot & Few-shot CoT) | ✅ |
| Self-Consistency (Majority Voting) | ✅ |
| Tree-of-Thought | ✅ |
| Role & Persona Engineering | ✅ |
| System Prompt Anatomy (ICONB framework) | ✅ |
| Multi-persona for Multi-agent | ✅ |
| JSON Mode & Structured Output | ✅ |
| Pydantic Validation | ✅ |
| Jinja2 Prompt Templates | ✅ |
| Prompt Chaining Pattern | ✅ |
| ReAct Pattern (Reason + Act) | ✅ |
| Meta-Prompting | ✅ |
| Self-Critique & Reflection | ✅ |
| Prompt Evaluation (LLM-as-judge) | ✅ |
| A/B Testing Framework | ✅ |
| Production Prompt Manager | ✅ |
| Common Anti-patterns | ✅ |

### 🗺️ Bước tiếp theo trong Roadmap:

```
Phase 1 (hoàn thành Prompt Engineering) → Tiếp theo:
├── Tokens & Context Windows
│   ├── Token counting với tiktoken
│   ├── Context management strategies
│   └── Summarization for long contexts
│
├── Embeddings
│   ├── text-embedding-3-small / large
│   ├── Cosine similarity
│   └── Semantic search
│
└── Streaming & Async
    ├── AsyncOpenAI
    ├── Streaming responses
    └── Concurrent API calls

Phase 2 → Tool Use · ReAct (production) · RAG · Vector DB (Chroma/Pinecone)
Phase 3 → LangChain · LangGraph · CrewAI · Deployment
```

### 💡 Key Takeaways từ 15 năm kinh nghiệm:

1. **Measure, don't guess** — Luôn có evaluation framework trước khi "optimize" prompt
2. **Production ≠ Playground** — Prompt tốt trong notebook có thể tệ ở production load
3. **Token là tiền** — Mỗi token tốn tiền; optimize prompt = optimize business cost
4. **Cache aggressively** — 30-50% requests thường là identical → cache = free money
5. **Model-specific tuning** — Không có "universal perfect prompt"; mỗi model có quirks riêng
6. **Chain > Monolith** — Prompt chains dễ debug, dễ optimize từng bước hơn là 1 prompt khổng lồ
7. **ReAct là foundation** — Hiểu ReAct = hiểu 80% của LangChain/LangGraph/AutoGen

In [ ]:
# Final: Quick reference card
print("""
╔══════════════════════════════════════════════════════════════╗
║          QUICK REFERENCE — Prompt Engineering Cheatsheet    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  WHEN TO USE WHICH TECHNIQUE:                                ║
║                                                              ║
║  Simple task           → Zero-shot                          ║
║  Custom format needed  → Few-shot (3-5 curated examples)    ║
║  Math/logic/reasoning  → + "Let's think step by step"       ║
║  High-stakes answer    → Self-Consistency (n=5-10)          ║
║  Complex decisions     → Tree-of-Thought                    ║
║  Specialist task       → Role Prompting (specific, not vague)║
║  Multi-step task       → Prompt Chaining                    ║
║  Tool use needed       → ReAct Pattern                      ║
║  Quality improvement   → Self-Critique pipeline             ║
║  Prompt optimization   → Meta-Prompting                     ║
║  Production system     → SmartPromptManager                 ║
║                                                              ║
║  GOLDEN RULES:                                               ║
║  1. Be specific, not vague                                   ║
║  2. Positive constraints > negative constraints              ║
║  3. Measure everything (A/B test)                            ║
║  4. Cache repeated calls                                     ║
║  5. Chain > Monolith for complex tasks                       ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")